## Viewing the DRS data and GLORI files 



### making the indexing of GLORI and DRS data uniform

In [3]:
import pandas as pd

#18 standard column names for modkit bedMethyl files
modkit_cols = [
    'chrom', 'start', 'end', 'modified_base', 'score', 'strand', 
    'thickStart', 'thickEnd', 'color', 'coverage', 'modified_percent', 
    'Nmod', 'Ncanonical', 'Nother', 'Ndelete', 'Nfail', 
    'Ndiff', 'Nvalid_cov'
]

#reading drs bedMethyl files
drs_df = pd.read_csv(
    "hsap_modkit_pileup_cov10.bed", 
    sep=r'\s+', 
    header=None, 
    names=modkit_cols,
    comment='#'
)

# Overwrite 'start' in-place: convert from 0-based to 1-based integer
drs_df['start'] = drs_df['start'].astype(int) + 1

# Convert numeric types in-place
drs_df['coverage'] = pd.to_numeric(drs_df['coverage'], errors='coerce')
drs_df['modified_percent'] = pd.to_numeric(drs_df['modified_percent'], errors='coerce')

print(f"Loaded {len(drs_df):,} rows successfully!")
print(drs_df.count)


Loaded 570,984 rows successfully!
<bound method DataFrame.count of                 chrom  start    end modified_base  score strand  thickStart  \
0                chr1  14415  14415             a     22      -       14414   
1                chr1  14517  14517             a     31      -       14516   
2                chr1  14569  14569             a     32      -       14568   
3                chr1  14586  14586             a     31      -       14585   
4                chr1  14614  14614             a     30      -       14613   
...               ...    ...    ...           ...    ...    ...         ...   
570979  chrKI270711.1  26346  26346             a     10      -       26345   
570980  chrKI270711.1  26356  26356             a     11      -       26355   
570981  chrKI270711.1  26374  26374             a     10      -       26373   
570982  chrKI270711.1  26465  26465             a     11      -       26464   
570983  chrKI270711.1  32030  32030             a     10      - 

### Union of GLORI positives and negatives CSV files

In [3]:
#loading GLORI files
glori_neg = pd.read_csv("same_transcript_negatives.csv")
glori_pos = pd.read_csv("same_transcript_positives.csv")


#renaming the seqnames column to chrom for standardization
for df in [glori_pos, glori_neg]:
    if 'seqnames' in df.columns:
        df.rename(columns={'seqnames': 'chrom'}, inplace=True)
    df['start'] = df['start'].astype(int)

# Assign explicit modification labels
glori_pos['glori_m6a'] = 'Modified'
glori_neg['glori_m6a'] = 'Unmodified'


#selecting key columns and labels
coord_cols = ['chrom', 'start', 'strand', 'glori_m6a']
glori_pos_sub = glori_pos[coord_cols].drop_duplicates()
glori_neg_sub = glori_neg[coord_cols].drop_duplicates()


#checking for conflicts and performing union of pos and negative CSVs
pos_keys = set(zip(glori_pos_sub['chrom'], glori_pos_sub['start'], glori_pos_sub['strand']))
neg_keys = set(zip(glori_neg_sub['chrom'], glori_neg_sub['start'], glori_neg_sub['strand']))

conflicting_keys = pos_keys.intersection(neg_keys)
print(f"Conflicting sites present in BOTH files: {len(conflicting_keys):,}")

#union
glori_union = pd.concat([glori_pos_sub, glori_neg_sub], ignore_index=True)
# Remove conflicting sites if any exist
if conflicting_keys:
    conflict_mask = glori_union.set_index(['chrom', 'start', 'strand']).index.isin(conflicting_keys)
    glori_union = glori_union[~conflict_mask].reset_index(drop=True)
    print(f"Removed {len(conflicting_keys):,} ambiguous sites present in both files.")

print(f"\nUnified GLORI reference ready!")
print(f"  - Total sites in Union: {len(glori_union):,}")
print(f"  - Modified (m6A):      {(glori_union['glori_m6a'] == 'Modified').sum():,}")
print(f"  - Unmodified (A):      {(glori_union['glori_m6a'] == 'Unmodified').sum():,}")


print(glori_union)

Conflicting sites present in BOTH files: 0

Unified GLORI reference ready!
  - Total sites in Union: 122,816
  - Modified (m6A):      61,408
  - Unmodified (A):      61,408
       chrom      start strand   glori_m6a
0       chr1     924038      +    Modified
1       chr1     924172      +    Modified
2       chr1     924469      +    Modified
3       chr1     924694      +    Modified
4       chr1     924760      +    Modified
...      ...        ...    ...         ...
122811  chrX  155458583      -  Unmodified
122812  chrX  155457251      -  Unmodified
122813  chrX  155524599      -  Unmodified
122814  chrX  155514032      -  Unmodified
122815  chrM      12662      +  Unmodified

[122816 rows x 4 columns]


### Intersection of DRS df and GLORI union

In [4]:
import pandas as pd

# 1. Total sites in raw inputs
total_drs_candidates = len(drs_df)
pos_glori_raw = len(glori_pos_sub)
neg_glori_raw = len(glori_neg_sub)
total_glori_union = len(glori_union)

# 2. Ensure coordinate types match before join
for df in [drs_df, glori_union]:
    df['chrom'] = df['chrom'].astype(str)
    df['start'] = df['start'].astype(int)
    df['strand'] = df['strand'].astype(str)

# 3. Perform the inner join
drs_labeled_df = pd.merge(
    drs_df,
    glori_union,
    on=['chrom', 'start', 'strand'],
    how='inner'
)

# 4. Breakdown of intersected sites
intersected_total = len(drs_labeled_df)
intersected_pos = (drs_labeled_df['glori_m6a'] == 'Modified').sum()
intersected_neg = (drs_labeled_df['glori_m6a'] == 'Unmodified').sum()

# 5. Print standard summary table
print("=" * 60)
print("              SITE INTERSECTION FUNNEL REPORT              ")
print("=" * 60)
print(f"1. RAW INPUTS")
print(f"   - DRS Candidate DRACH Sites (Modkit):    {total_drs_candidates:,}")
print(f"   - GLORI Positive Sites (Modified):       {pos_glori_raw:,}")
print(f"   - GLORI Negative Sites (Unmodified):     {neg_glori_raw:,}")
print(f"   - Deduplicated Clean GLORI Reference:    {total_glori_union:,}")
print("-" * 60)
print(f"2. INTERSECTION RESULTS (drs_df ∩ glori_union)")
print(f"   - Total Intersected Ground-Truth Sites:  {intersected_total:,}")
print(f"     ├── True Positives  (Modified m6A):    {intersected_pos:,} ({(intersected_pos/intersected_total)*100:.2f}%)")
print(f"     └── True Negatives  (Unmodified A):    {intersected_neg:,} ({(intersected_neg/intersected_total)*100:.2f}%)")
print("-" * 60)
print(f"3. COVERAGE / MATCH RATES")
print(f"   - DRS Candidates with GLORI Label:        {(intersected_total/total_drs_candidates)*100:.2f}%")
print(f"   - GLORI Reference Recovered in DRS:       {(intersected_total/total_glori_union)*100:.2f}%")
print("=" * 60)

              SITE INTERSECTION FUNNEL REPORT              
1. RAW INPUTS
   - DRS Candidate DRACH Sites (Modkit):    570,984
   - GLORI Positive Sites (Modified):       61,408
   - GLORI Negative Sites (Unmodified):     61,408
   - Deduplicated Clean GLORI Reference:    122,816
------------------------------------------------------------
2. INTERSECTION RESULTS (drs_df ∩ glori_union)
   - Total Intersected Ground-Truth Sites:  45,836
     ├── True Positives  (Modified m6A):    43,775 (95.50%)
     └── True Negatives  (Unmodified A):    2,061 (4.50%)
------------------------------------------------------------
3. COVERAGE / MATCH RATES
   - DRS Candidates with GLORI Label:        8.03%
   - GLORI Reference Recovered in DRS:       37.32%


In [5]:
print(glori_union.head())

  chrom   start strand glori_m6a
0  chr1  924038      +  Modified
1  chr1  924172      +  Modified
2  chr1  924469      +  Modified
3  chr1  924694      +  Modified
4  chr1  924760      +  Modified


### Checking if DRS detected the highest-methylated GLORI sites

In [6]:
#identifying GLORI sites detected vs missed in DRS
glori_pos_sub['in_drs'] = glori_pos_sub.set_index(['chrom', 'start', 'strand']).index.isin(
    drs_df.set_index(['chrom', 'start', 'strand']).index
)

#comparing GLORI methylation rates between matched and missed sites w DRS
#mapping the new in_drs column to original glori_pos dataframe
glori_pos['in_drs'] = glori_pos_sub['in_drs'].values


mean_ratio_detected = glori_pos[glori_pos['in_drs']]['Ratio_mean'].mean()
mean_ratio_missed   = glori_pos[~glori_pos['in_drs']]['Ratio_mean'].mean() 
print("=== METHYLATION RATE COMPARISON (GLORI Ratio_mean) ===")
print(f"Mean methylation rate for sites DETECTED in DRS: {mean_ratio_detected:.3f}")
print(f"Mean methylation rate for sites MISSED in DRS:   {mean_ratio_missed:.3f}\n")


#comparing GLORI coverage in missed and detected sites
mean_cov_detected = glori_pos[glori_pos['in_drs']]['coverage_geomean'].mean()
mean_cov_missed   = glori_pos[~glori_pos['in_drs']]['coverage_geomean'].mean()
print("=== COVERAGE COMPARISON (GLORI coverage_geomean) ===")
print(f"Mean GLORI coverage for sites DETECTED in DRS: {mean_cov_detected:.1f}x")
print(f"Mean GLORI coverage for sites MISSED in DRS:   {mean_cov_missed:.1f}x")

=== METHYLATION RATE COMPARISON (GLORI Ratio_mean) ===
Mean methylation rate for sites DETECTED in DRS: 0.448
Mean methylation rate for sites MISSED in DRS:   0.370

=== COVERAGE COMPARISON (GLORI coverage_geomean) ===
Mean GLORI coverage for sites DETECTED in DRS: 446.5x
Mean GLORI coverage for sites MISSED in DRS:   284.6x


### Sampling 2,061 positive sites to match 2061 negative sites

In [7]:
#isolating the true negative sites detected in DRS
drs_negatives = drs_labeled_df[drs_labeled_df['glori_m6a'] == 'Unmodified'].copy()

#sampling 2,061 true positive sites from available sites in DRS
drs_positives_sampled = drs_labeled_df[drs_labeled_df['glori_m6a'] == 'Modified'].sample(
    n=len(drs_negatives), 
    random_state=42
)

#combining into a balanced df
balanced_df = pd.concat([drs_positives_sampled, drs_negatives], ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
balanced_df.to_csv("balanced_ground_truth_4k.tsv", sep="\t", index=False)

print(balanced_df['glori_m6a'].value_counts())

glori_m6a
Modified      2061
Unmodified    2061
Name: count, dtype: int64


In [8]:
print()

### Creating the exact 4122 target sites 0-based BED file

In [8]:
import pandas as pd

# 1. Load ground truth TSV
df = pd.read_csv("balanced_ground_truth_4k.tsv", sep="\t")

# 2. Convert chrom to HSAP_ format (e.g., 'chr6' or '6' -> 'HSAP_6')
# Strips 'chr' if present, then prefixes with 'HSAP_'
df['chrom_hsap'] = "HSAP_" + df['chrom'].astype(str).str.replace('^chr', '', regex=True)

# 3. Use thickStart (0-based) and end (1-based)
df['start_0based'] = df['thickStart']
df['end_1based'] = df['end']

# 4. Create unique site identifier
df['site_id'] = (
    df['chrom_hsap'].astype(str) + ":" + 
    df['start'].astype(str) + ":" + 
    df['strand'].astype(str)
)

df['score'] = 0

# 5. Build 6-column BED
bed_df = df[['chrom_hsap', 'start_0based', 'end_1based', 'site_id', 'score', 'strand']]

# Save over target_sites_4k.bed
bed_df.to_csv("target_sites_4k.bed", sep="\t", index=False, header=False)
print("Updated target_sites_4k.bed! First 3 rows:")
print(bed_df.head(3))

Updated target_sites_4k.bed! First 3 rows:
  chrom_hsap  start_0based  end_1based             site_id  score strand
0     HSAP_6     108684140   108684141  HSAP_6:108684141:+      0      +
1     HSAP_5     176658102   176658103  HSAP_5:176658103:+      0      +
2    HSAP_11      70206693    70206694  HSAP_11:70206694:+      0      +


### merging per read mod calls summary statistics as columns to our main dataframe

In [9]:
import numpy as np

#loading files
gt_df = pd.read_csv("balanced_ground_truth_4k.tsv", sep="\t")
mod_df = pd.read_csv("per_read_mod_calls_4k.tsv", sep="\t")

print(f"Ground Truth shape: {gt_df.shape}")
print(f"Per-read calls shape: {mod_df.shape}\n")

print("Ground Truth columns:", list(gt_df.columns))
print("Per-read calls columns:", list(mod_df.columns))



#subseting the ground truth df
gt_cols = ['chrom', 'start', 'end', 'score', 'strand', 'coverage', 'modified_percent', 'glori_m6a']
gt_sub = gt_df[[c for c in gt_cols if c in gt_df.columns]].copy()

# Strip 'chr' prefix -> turns 'chr1' into '1'
gt_sub['chrom_key'] = gt_sub['chrom'].astype(str).str.replace('^chr', '', regex=True)

mod_clean = mod_df[
    (mod_df['ref_position'] != -1) & 
    (mod_df['mod_code'].isin(['a', '21839', 'm6A', 'a_m6A']))
].copy()

# Strip 'HSAP_' prefix -> turns 'HSAP_1' into '1'
mod_clean['chrom_key'] = mod_clean['chrom'].astype(str).str.replace('^HSAP_', '', regex=True)
# Convert 0-based ref_position to 1-based start coordinate
mod_clean['start_1based'] = mod_clean['ref_position'] + 1

print("GT chrom_key sample:", gt_sub['chrom_key'].head(3).tolist())
print("Modkit chrom_key sample:", mod_clean['chrom_key'].head(3).tolist())

strand_col = 'ref_mod_strand' if 'ref_mod_strand' in mod_clean.columns else 'ref_strand'

#  Standardize mod_qual to continuous [0.0, 1.0] probability scale
# If mod_qual max exceeds 1.0 (e.g. 0-255 scale), divide by 255.0; otherwise leave as float
if mod_clean['mod_qual'].max() > 1.0:
    mod_clean['mod_prob'] = mod_clean['mod_qual'] / 255.0
else:
    mod_clean['mod_prob'] = mod_clean['mod_qual']

# Group by locus keys (chrom_key, start_1based, strand) and aggregate mod_prob
grouped = mod_clean.groupby(['chrom_key', 'start_1based', strand_col])['mod_prob']

stats_df = grouped.agg(
    mod_prob_median=('median'),
    mod_prob_q25=lambda x: np.percentile(x, 25),
    mod_prob_q75=lambda x: np.percentile(x, 75),
    mod_prob_sd=('std')
).reset_index()

# Handle single-read loci where std returns NaN
stats_df['mod_prob_sd'] = stats_df['mod_prob_sd'].fillna(0.0)

#Merge summary features onto ground truth subset
master_df = pd.merge(
    gt_sub,
    stats_df,
    left_on=['chrom_key', 'start', 'strand'],
    right_on=['chrom_key', 'start_1based', strand_col],
    how='left'
)

#  Drop temporary merge keys
master_df = master_df.drop(columns=['chrom_key', 'start_1based', strand_col], errors='ignore')

for col in ['mod_prob_median', 'mod_prob_q25', 'mod_prob_q75', 'mod_prob_sd']:
    master_df[col] = master_df[col].fillna(0.0)

# Print verification details
print("\n--- Merge Complete ---")
print(f"Master dataframe shape: {master_df.shape}")
print(f"Columns ({len(master_df.columns)}): {list(master_df.columns)}")
# Display top 5 rows
display(master_df.head())


Ground Truth shape: (4122, 19)
Per-read calls shape: (336903, 19)

Ground Truth columns: ['chrom', 'start', 'end', 'modified_base', 'score', 'strand', 'thickStart', 'thickEnd', 'color', 'coverage', 'modified_percent', 'Nmod', 'Ncanonical', 'Nother', 'Ndelete', 'Nfail', 'Ndiff', 'Nvalid_cov', 'glori_m6a']
Per-read calls columns: ['read_id', 'forward_read_position', 'ref_position', 'chrom', 'mod_strand', 'ref_strand', 'ref_mod_strand', 'fw_soft_clipped_start', 'fw_soft_clipped_end', 'read_length', 'mod_qual', 'mod_code', 'base_qual', 'ref_kmer', 'query_kmer', 'canonical_base', 'modified_primary_base', 'inferred', 'flag']
GT chrom_key sample: ['6', '5', '11']
Modkit chrom_key sample: ['1', '1', '1']

--- Merge Complete ---
Master dataframe shape: (4122, 12)
Columns (12): ['chrom', 'start', 'end', 'score', 'strand', 'coverage', 'modified_percent', 'glori_m6a', 'mod_prob_median', 'mod_prob_q25', 'mod_prob_q75', 'mod_prob_sd']


,chrom,start,end,score,strand,coverage,modified_percent,glori_m6a,mod_prob_median,mod_prob_q25,mod_prob_q75,mod_prob_sd
0,chr6,108684141,108684141,50,+,50,26.00,Modified,0.001953,0.001953,0.739258,0.434578
1,chr5,176658103,176658103,39,+,39,84.62,Modified,0.998047,0.991211,0.998047,0.375025
2,chr11,70206694,70206694,80,+,80,70.00,Modified,0.998047,0.005859,0.998047,0.447917
3,chr17,32941254,32941254,93,+,93,53.76,Modified,0.958984,0.001953,0.998047,0.487936
4,chr3,131383499,131383499,38,+,38,36.84,Modified,0.005859,0.001953,0.998047,0.484049


### Base quality scores statistics 

In [10]:
import pandas as pd
import numpy as np

# 1. Load files
gt_df = pd.read_csv("balanced_ground_truth_4k.tsv", sep="\t")
mod_df = pd.read_csv("per_read_mod_calls_4k.tsv", sep="\t")

# 2. Prepare GT subset
gt_cols = ['chrom', 'start', 'end', 'score', 'strand', 'coverage', 'modified_percent', 'glori_m6a']
gt_sub = gt_df[[c for c in gt_cols if c in gt_df.columns]].copy()
gt_sub['chrom_key'] = gt_sub['chrom'].astype(str).str.replace('^chr', '', regex=True)

# 3. Clean Modkit per-read calls
mod_clean = mod_df[
    (mod_df['ref_position'] != -1) & 
    (mod_df['mod_code'].isin(['a', '21839', 'm6A', 'a_m6A']))
].copy()

mod_clean['chrom_key'] = mod_clean['chrom'].astype(str).str.replace('^HSAP_', '', regex=True)
mod_clean['start_1based'] = mod_clean['ref_position'] + 1
strand_col = 'ref_mod_strand' if 'ref_mod_strand' in mod_clean.columns else 'ref_strand'

# Standardize mod_qual to continuous [0.0, 1.0] probability scale
if mod_clean['mod_qual'].max() > 1.0:
    mod_clean['mod_prob'] = mod_clean['mod_qual'] / 255.0
else:
    mod_clean['mod_prob'] = mod_clean['mod_qual']

# 4. Aggregate BOTH mod_prob and base_qual statistics in a single pass
stats_df = mod_clean.groupby(['chrom_key', 'start_1based', strand_col]).agg(
    mod_prob_median=('mod_prob', 'median'),
    mod_prob_q25=('mod_prob', lambda x: np.percentile(x, 25)),
    mod_prob_q75=('mod_prob', lambda x: np.percentile(x, 75)),
    mod_prob_sd=('mod_prob', 'std'),
   # Base quality quartile features (Q25, Q50/Median, Q75)
    base_qual_q25=('base_qual', lambda x: np.percentile(x, 25)),
    base_qual_q50=('base_qual', 'median'),
    base_qual_q75=('base_qual', lambda x: np.percentile(x, 75))
).reset_index()

# Handle single-read loci where std returns NaN
stats_df['mod_prob_sd'] = stats_df['mod_prob_sd'].fillna(0.0)

# 5. Merge onto GT subset
master_df = pd.merge(
    gt_sub,
    stats_df,
    left_on=['chrom_key', 'start', 'strand'],
    right_on=['chrom_key', 'start_1based', strand_col],
    how='left'
)

# 6. Drop temporary merge keys
master_df = master_df.drop(columns=['chrom_key', 'start_1based', strand_col], errors='ignore')

# 7. Fill default zeros for uncovered sites
feature_cols = [
    'mod_prob_median', 'mod_prob_q25', 'mod_prob_q75', 'mod_prob_sd',
    'base_qual_q25', 'base_qual_q50', 'base_qual_q75'
]
for col in feature_cols:
    master_df[col] = master_df[col].fillna(0.0)

# 8. Print verification details & export
print("\n--- Pipeline Complete ---")
print(f"Master dataframe shape: {master_df.shape}")
print(f"Columns ({len(master_df.columns)}): {list(master_df.columns)}")

display(master_df.head())
master_df.to_csv("master_df_temp.tsv", sep="\t", index=False)


--- Pipeline Complete ---
Master dataframe shape: (4122, 15)
Columns (15): ['chrom', 'start', 'end', 'score', 'strand', 'coverage', 'modified_percent', 'glori_m6a', 'mod_prob_median', 'mod_prob_q25', 'mod_prob_q75', 'mod_prob_sd', 'base_qual_q25', 'base_qual_q50', 'base_qual_q75']


,chrom,start,end,score,strand,coverage,modified_percent,glori_m6a,mod_prob_median,mod_prob_q25,mod_prob_q75,mod_prob_sd,base_qual_q25,base_qual_q50,base_qual_q75
0,chr6,108684141,108684141,50,+,50,26.00,Modified,0.001953,0.001953,0.739258,0.434578,30.00,31.0,32.00
1,chr5,176658103,176658103,39,+,39,84.62,Modified,0.998047,0.991211,0.998047,0.375025,30.00,32.0,33.75
2,chr11,70206694,70206694,80,+,80,70.00,Modified,0.998047,0.005859,0.998047,0.447917,23.00,30.0,32.00
3,chr17,32941254,32941254,93,+,93,53.76,Modified,0.958984,0.001953,0.998047,0.487936,28.00,29.0,30.00
4,chr3,131383499,131383499,38,+,38,36.84,Modified,0.005859,0.001953,0.998047,0.484049,15.75,28.0,29.25


In [11]:
import pandas as pd

# Load finalized dataset
final_df = pd.read_csv("final_m6a_feature_matrix.tsv", sep="\t")

print(f"Final Dataset Shape: {final_df.shape}")
print(f"Total Columns ({len(final_df.columns)}): {list(final_df.columns)}\n")

display(final_df.head(10))
print(final_df.shape)
print((final_df["glori_m6a"] == "Modified").sum())
print((final_df["glori_m6a"] == "Unmodified").sum())


Final Dataset Shape: (4122, 20)
Total Columns (20): ['chrom', 'start', 'end', 'score', 'strand', 'coverage', 'modified_percent', 'glori_m6a', 'mod_prob_median', 'mod_prob_q25', 'mod_prob_q75', 'mod_prob_sd', 'base_qual_q25', 'base_qual_q50', 'base_qual_q75', 'kmer_5mer', 'pos_minus2', 'pos_minus1', 'pos_plus1', 'pos_plus2']



,chrom,start,end,score,strand,coverage,modified_percent,glori_m6a,mod_prob_median,mod_prob_q25,mod_prob_q75,mod_prob_sd,base_qual_q25,base_qual_q50,base_qual_q75,kmer_5mer,pos_minus2,pos_minus1,pos_plus1,pos_plus2
0,chr6,108684141,108684141,50,+,50,26.00,Modified,0.001953,0.001953,0.739258,0.434578,30.00,31.0,32.00,GAACT,G,A,C,T
1,chr5,176658103,176658103,39,+,39,84.62,Modified,0.998047,0.991211,0.998047,0.375025,30.00,32.0,33.75,GGACT,G,G,C,T
2,chr11,70206694,70206694,80,+,80,70.00,Modified,0.998047,0.005859,0.998047,0.447917,23.00,30.0,32.00,TGACA,T,G,C,A
3,chr17,32941254,32941254,93,+,93,53.76,Modified,0.958984,0.001953,0.998047,0.487936,28.00,29.0,30.00,GGACT,G,G,C,T
4,chr3,131383499,131383499,38,+,38,36.84,Modified,0.005859,0.001953,0.998047,0.484049,15.75,28.0,29.25,GGACC,G,G,C,C
5,chr19,46410591,46410591,31,-,31,100.00,Modified,0.998047,0.994141,0.998047,0.067871,28.00,33.0,35.00,GGACT,G,G,C,T
6,chr5,66053837,66053837,21,+,21,14.29,Modified,0.001953,0.001953,0.001953,0.356866,34.00,35.0,36.00,TGACA,T,G,C,A
7,chr6,33455866,33455866,22,+,22,63.64,Modified,0.972656,0.025391,0.998047,0.470572,27.25,32.5,33.75,TGACT,T,G,C,T
8,chr17,40171058,40171058,72,+,72,0.00,Unmodified,0.001953,0.001953,0.005859,0.062901,16.00,29.0,32.00,TAACT,T,A,C,T
9,chr17,75273439,75273439,81,-,81,23.46,Modified,0.001953,0.001953,0.105469,0.411314,27.00,31.0,32.00,GGACC,G,G,C,C


(4122, 20)
2061
2061


### Preparing test and training/validation datasets


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold

# 1. Load full feature matrix
df = pd.read_csv("final_m6a_feature_matrix.tsv", sep="\t")

target_col = 'glori_m6a'
# Map string labels to numeric binary integers (1 for Modified, 0 for Unmodified)
label_map = {'Modified': 1, 'Unmodified': 0}
df[target_col] = df[target_col].map(label_map)

# 2. One-hot encode positional sequence features (-2, -1, +1, +2)
seq_cols = ['pos_minus2', 'pos_minus1', 'pos_plus1', 'pos_plus2']
df_encoded = pd.get_dummies(df, columns=seq_cols, drop_first=False)

# 3. Define feature columns (exclude non-feature metadata and target)
metadata_cols = ['chrom', 'start', 'end', 'score', 'strand', 'kmer_5mer', target_col]
feature_cols = [c for c in df_encoded.columns if c not in metadata_cols]

X = df_encoded[feature_cols].copy()
y = df_encoded[target_col].values

# 4. Stratified Holdout Test Split (20% = 824 sites)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

print(f"Total dataset: {len(df)} sites")
print(f"Holdout Test Set: {len(X_test)} sites (Pos: {sum(y_test==1)}, Neg: {sum(y_test==0)})")
print(f"Train/Val Pool: {len(X_train_val)} sites (Pos: {sum(y_train_val==1)}, Neg: {sum(y_train_val==0)})\n")

# 5. Build 5 Stratified Folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = list(skf.split(X_train_val, y_train_val))

for i, (train_idx, val_idx) in enumerate(folds, 1):
    val_pos = sum(y_train_val[val_idx] == 1)
    val_neg = sum(y_train_val[val_idx] == 0)
    print(f"Fold {i}: Train = {len(train_idx)} sites | Val = {len(val_idx)} sites (Pos: {val_pos}, Neg: {val_neg})")



Total dataset: 4122 sites
Holdout Test Set: 825 sites (Pos: 412, Neg: 413)
Train/Val Pool: 3297 sites (Pos: 1649, Neg: 1648)

Fold 1: Train = 2637 sites | Val = 660 sites (Pos: 330, Neg: 330)
Fold 2: Train = 2637 sites | Val = 660 sites (Pos: 330, Neg: 330)
Fold 3: Train = 2638 sites | Val = 659 sites (Pos: 330, Neg: 329)
Fold 4: Train = 2638 sites | Val = 659 sites (Pos: 330, Neg: 329)
Fold 5: Train = 2638 sites | Val = 659 sites (Pos: 329, Neg: 330)


### Exporting test and training sets

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load raw feature matrix
df = pd.read_csv("final_m6a_feature_matrix.tsv", sep="\t")

target_col = 'glori_m6a'

# Map string labels: Modified -> 1, Unmodified -> 0
label_map = {'Modified': 1, 'Unmodified': 0}
df[target_col] = df[target_col].map(label_map)

# 2. One-hot encode sequence context positions
seq_cols = ['pos_minus2', 'pos_minus1', 'pos_plus1', 'pos_plus2']
df_encoded = pd.get_dummies(df, columns=[c for c in seq_cols if c in df.columns], drop_first=False)

# 3. Drop non-numeric metadata text columns entirely
metadata_cols = ['chrom', 'start', 'end', 'score', 'strand', 'kmer_5mer']
clean_df = df_encoded.drop(columns=[c for c in metadata_cols if c in df_encoded.columns])

# Separate X (features) and y (target)
X = clean_df.drop(columns=[target_col]).astype(float)
y = clean_df[target_col].values

# 4. Stratified Holdout Test Split (20% test = 825 sites, 80% train/val = 3,297 sites)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 5. Save as clean, numeric-only TSVs
train_val_df = X_train_val.copy()
train_val_df[target_col] = y_train_val

test_df = X_test.copy()
test_df[target_col] = y_test

train_val_df.to_csv("train_val_matrix.tsv", sep="\t", index=False)
test_df.to_csv("test_matrix.tsv", sep="\t", index=False)

print("✓ Clean Export Complete:")
print(f" - train_val_matrix.tsv: {train_val_df.shape[0]} rows × {train_val_df.shape[1]} numeric columns")
print(f" - test_matrix.tsv:      {test_df.shape[0]} rows × {test_df.shape[1]} numeric columns")

✓ Clean Export Complete:
 - train_val_matrix.tsv: 3297 rows × 19 numeric columns
 - test_matrix.tsv:      825 rows × 19 numeric columns


### reconstructing training and test sets without modified_fraction label

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load raw feature matrix
df = pd.read_csv("final_m6a_feature_matrix.tsv", sep="\t")

target_col = 'glori_m6a'

# Map string labels: Modified -> 1, Unmodified -> 0
label_map = {'Modified': 1, 'Unmodified': 0}
df[target_col] = df[target_col].map(label_map)

# 2. One-hot encode sequence context positions
seq_cols = ['pos_minus2', 'pos_minus1', 'pos_plus1', 'pos_plus2']
df_encoded = pd.get_dummies(df, columns=[c for c in seq_cols if c in df.columns], drop_first=False)

# 3. Drop metadata AND target-derived summary metrics (prevent leakage)
leakage_cols = [
    'modified_percent',  # Primary leakage source
    'chrom', 'start', 'end', 'score', 'strand', 'kmer_5mer'
]
clean_df = df_encoded.drop(columns=[c for c in leakage_cols if c in df_encoded.columns])

# Separate X (features) and y (target)
X = clean_df.drop(columns=[target_col]).astype(float)
y = clean_df[target_col].values

# 4. Stratified Holdout Test Split (20% test = 825 sites, 80% train/val = 3,297 sites)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 5. Save clean numeric TSVs
train_val_df = X_train_val.copy()
train_val_df[target_col] = y_train_val

test_df = X_test.copy()
test_df[target_col] = y_test

train_val_df.to_csv("train_val_matrix2.tsv", sep="\t", index=False)
test_df.to_csv("test_matrix2.tsv", sep="\t", index=False)

print("✓ Re-export Complete (Leakage Removed):")
print(f" - train_val_matrix2.tsv: {train_val_df.shape[0]} rows × {train_val_df.shape[1]} columns")
print(f" - test_matrix2.tsv:      {test_df.shape[0]} rows × {test_df.shape[1]} columns")

✓ Re-export Complete (Leakage Removed):
 - train_val_matrix2.tsv: 3297 rows × 18 columns
 - test_matrix2.tsv:      825 rows × 18 columns


In [17]:
import pandas as pd
df = pd.read_csv("train_val_matrix2.tsv", sep="\t")
print(f"Total columns: {df.shape[1]}")
print(df.columns.tolist())

Total columns: 18
['coverage', 'mod_prob_median', 'mod_prob_q25', 'mod_prob_q75', 'mod_prob_sd', 'base_qual_q25', 'base_qual_q50', 'base_qual_q75', 'pos_minus2_A', 'pos_minus2_G', 'pos_minus2_T', 'pos_minus1_A', 'pos_minus1_G', 'pos_plus1_C', 'pos_plus2_A', 'pos_plus2_C', 'pos_plus2_T', 'glori_m6a']


### Performing chromosome based training and test set split

In [ ]:
import pandas as pd
df = pd.read_csv("final_m6a_feature_matrix.tsv", sep="\t")

# Standardize target label (1 = Modified, 0 = Unmodified)
if "target" not in df.columns:
    # Convert to string and strip whitespace to be safe
    raw_col = df["glori_m6a"].astype(str).str.strip().str.lower()
    
    # Map all common variations to 1 and 0
    mapping = {
        "modified": 1, "1": 1, "1.0": 1,
        "unmodified": 0, "0": 0, "0.0": 0
    }
    df["target"] = raw_col.map(mapping)

##SORTING THE SITES BASED ON CHROMOSOMES
# Custom chromosome sorting (chr1-chr22, chrX, chrY, chrM)
def chrom_sort_key(chrom_str):
    c = chrom_str.replace("chr", "")
    if c.isdigit():
        return (0, int(c))
    elif c == "X":
        return (1, 23)
    elif c == "Y":
        return (1, 24)
    elif c in ["M", "MT"]:
        return (1, 25)
    else:
        return (2, c)

df["chrom_sort"] = df["chrom"].apply(chrom_sort_key)
df = df.sort_values(by=["chrom_sort", "start"]).drop(columns=["chrom_sort"]).reset_index(drop=True)


##INSPECTING SITES AND CLASS DISTRIBUTIONS FOR EACH CHROMOSOMES
chrom_summary = df.groupby("chrom").agg(
    total_sites=("target", "count"),
    positive_1s=("target", lambda x: (x == 1).sum()),
    negative_0s=("target", lambda x: (x == 0).sum())
).reset_index()

print(chrom_summary)


    chrom  total_sites  positive_1s  negative_0s
0    chr1          462          247          215
1   chr10          172           83           89
2   chr11          188           99           89
3   chr12          200           95          105
4   chr13           58           32           26
5   chr14          140           76           64
6   chr15          132           67           65
7   chr16          259          131          128
8   chr17          315          145          170
9   chr18           36           18           18
10  chr19          228          119          109
11   chr2          268          130          138
12  chr20          112           54           58
13  chr21           55           24           31
14  chr22          152           76           76
15   chr3          225          110          115
16   chr4          113           56           57
17   chr5          159           81           78
18   chr6          231          115          116
19   chr7          1

1. Target MetricsTotal Dataset ($N$): $4,122$ sitesTotal Positives ($1$): $2,061$ ($50.0\%$)Total Negatives ($0$): $2,061$ ($50.0\%$)Target Test Set ($\sim 20\%$): $\sim 824$ sitesTarget Positives: $\sim 412$ sitesTarget Negatives: $\sim 412$ sites

2. Optimal Test Chromosome CombinationTo hit almost exactly $20\%$ of the data while maintaining a $1:1$ positive-to-negative ratio, the best combination of whole chromosomes to hold out for the test set is:$$\mathbf{\text{Test Chromosomes} = [\text{chr1}, \text{chr17}, \text{chr21}]}$$


Individual Breakdown:chr1: $462$ sites ($247$ pos, $215$ neg)chr17: $315$ sites ($145$ pos, $170$ neg)chr21: $55$ sites ($24$ pos, $31$ neg)

In [ ]:
import pandas as pd

# 0. Load your main intersected dataframe (adjust path/variable as needed)
# df = pd.read_csv("your_intersected_dataset.tsv", sep="\t")

# Ensure target label is mapped properly across df first
if "target" not in df.columns:
    raw_col = df["glori_m6a"].astype(str).str.strip().str.lower()
    mapping = {"modified": 1, "1": 1, "1.0": 1, "unmodified": 0, "0": 0, "0.0": 0}
    df["target"] = raw_col.map(mapping)

# DEFINE CHOSEN TEST CHROMOSOMES
TEST_CHROMOSOMES = ["chr1", "chr17", "chr21"]

# Perform Split
test_df      = df[df["chrom"].isin(TEST_CHROMOSOMES)].copy().reset_index(drop=True)
train_val_df = df[~df["chrom"].isin(TEST_CHROMOSOMES)].copy().reset_index(drop=True)

# Context encoding function
context_cols = ["pos_minus2", "pos_minus1", "pos_plus2", "pos_plus1"]
BASES = ["A", "C", "G", "T"]

def encode_context(frame):
    frame = frame.copy()
    for col in context_cols:
        if col in frame.columns:
            frame[col] = (frame[col].astype(str).str.strip().str.upper()
                          .replace({"U": "T"}))
            frame[col] = frame[col].where(frame[col].isin(BASES), "N")
            frame[col] = pd.Categorical(frame[col], categories=BASES)
    
    existing_context = [c for c in context_cols if c in frame.columns]
    if existing_context:
        frame = pd.get_dummies(frame, columns=existing_context, prefix=existing_context, dtype=int)
    if "kmer_5mer" in frame.columns:
        frame = frame.drop(columns="kmer_5mer")
    return frame

# Encode both sets completely
train_val_enc = encode_context(train_val_df)
test_enc      = encode_context(test_df)

# Define probability columns block
prob_cols = ["mod_prob_median", "mod_prob_sd", "mod_prob_q25", "mod_prob_q75"]




# MODEL 1: NO PROBABILITY FEATURES (Genomic/Sequence features only)
# ==========================================
drop_cols_m1 = ["chrom", "start", "end", "kmer_5mer", "glori_m6a", "strand", 
                "modified_percent"] + prob_cols

train_m1 = train_val_enc.drop(columns=[c for c in drop_cols_m1 if c in train_val_enc.columns])


train_m1.to_csv("final_train_set_model1.tsv", sep="\t", index=False)



# MODEL 2: ONLY PROBABILITY FEATURES + TARGET
# ==========================================
features_m2 = prob_cols + ["target"]

train_m2 = train_val_enc[[c for c in features_m2 if c in train_val_enc.columns]].copy()

train_m2.to_csv("final_train_set_model2.tsv", sep="\t", index=False)



# MODEL 3: ALL FEATURES EXCEPT MOD FRACTION ("modified_percent")
# ==========================================
drop_cols_m3 = ["chrom", "start", "end", "kmer_5mer", "glori_m6a", "strand", "modified_percent"]

train_m3 = train_val_enc.drop(columns=[c for c in drop_cols_m3 if c in train_val_enc.columns])

train_m3.to_csv("final_train_set_model3.tsv", sep="\t", index=False)


In [ ]:
train_val_enc[["chrom", "start", "end"]].to_csv("train_val_ids.csv", index=False)
